# Analysis: Evaluation, Metrics & Insights

**Purpose**: Deep dive into model performance, failure modes, and improvements.

**Prerequisites**:
- Run all previous notebooks (pipeline, training, inference)

**Workflow**:
1. Analyze per-class metrics (precision, recall, F1)
2. Identify failure cases (false positives, false negatives)
3. Visualize challenging samples
4. Compute confusion matrices
5. Generate recommendations for improvement

**Outputs**:
- Per-class performance breakdown
- Visualizations of failure modes
- Insights for hyperparameter tuning
- Recommendations for next iterations

**Issues**: #14 Hyperparameter Tuning (D4), #15 Loss Optimization (D4), #16 Cross-Validation (D4)

## Setup

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import json
from pathlib import Path
from sklearn.metrics import (
    confusion_matrix, precision_recall_fscore_support, 
    roc_curve, auc, roc_auc_score, accuracy_score
)

# Mount Google Drive
import os
if not os.path.exists('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')
    print("✓ Google Drive mounted")
else:
    print("✓ Google Drive already mounted")

# Setup paths
predictions_dir = Path('inference_results')
output_dir = Path('inference_results')
output_dir.mkdir(exist_ok=True)

drive_results_dir = Path('/content/drive/MyDrive/RETINNA_checkpoints/analysis_results')
drive_results_dir.mkdir(parents=True, exist_ok=True)

print(f"✓ Output directory: {output_dir}")
print(f"✓ Drive backup directory: {drive_results_dir}")

## Load Predictions from Drive

In [ ]:
# Load predictions
predictions_path = Path('inference_results/predictions.pt')

# If predictions don't exist locally, copy from Drive
if not predictions_path.exists():
    drive_predictions = Path('/content/drive/MyDrive/RETINNA_checkpoints/predictions.pt')
    if drive_predictions.exists():
        print("Predictions not found locally, copying from Google Drive...")
        import shutil
        shutil.copy(drive_predictions, predictions_path)
        print(f"✓ Copied from {drive_predictions}")
    else:
        print(f"Error: Predictions not found at {predictions_path} or {drive_predictions}")
        print("Make sure to run 04_inference.ipynb first.")

# Load predictions
if predictions_path.exists():
    print(f"\nLoading predictions from {predictions_path}...")
    data = torch.load(predictions_path)
    predictions = data['predictions']  # [N, H, W] - probabilities
    targets = data['targets']  # [N, H, W] - binary masks
    images = data['images']  # [N, C, H, W] - pre-fire images
    print(f"✓ Loaded predictions: {predictions.shape}")
    print(f"✓ Loaded targets: {targets.shape}")
    print(f"✓ Loaded images: {images.shape}")
else:
    print("Could not load predictions - exiting")

## Compute Metrics

In [ ]:
# Convert to binary predictions at threshold 0.5
threshold = 0.5
pred_binary = (predictions > threshold).float()

# Flatten for metric computation
pred_flat = pred_binary.numpy().flatten()
target_flat = targets.numpy().flatten()
pred_prob_flat = predictions.numpy().flatten()

# Compute metrics
accuracy = accuracy_score(target_flat, pred_flat)
precision, recall, f1, _ = precision_recall_fscore_support(target_flat, pred_flat, average='binary')

# IoU (Intersection over Union)
intersection = np.sum(pred_flat * target_flat)
union = np.sum(pred_flat + target_flat) - intersection
iou = intersection / (union + 1e-8)

# ROC AUC
roc_auc = roc_auc_score(target_flat, pred_prob_flat)

# Confusion matrix
cm = confusion_matrix(target_flat, pred_flat)
tn, fp, fn, tp = cm.ravel()

# Compile metrics
metrics = {
    "threshold": threshold,
    "accuracy": float(accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "f1_score": float(f1),
    "iou": float(iou),
    "roc_auc": float(roc_auc),
    "true_negatives": int(tn),
    "false_positives": int(fp),
    "false_negatives": int(fn),
    "true_positives": int(tp),
    "specificity": float(tn / (tn + fp)) if (tn + fp) > 0 else 0,
    "sensitivity": float(recall),
    "test_samples": len(predictions),
    "total_pixels": len(pred_flat),
    "burned_pixels_ground_truth": int(np.sum(target_flat)),
    "burned_pixels_predicted": int(np.sum(pred_flat))
}

print("=== Test Set Metrics (Threshold 0.5) ===")
print(f"Accuracy:     {metrics['accuracy']:.4f}")
print(f"Precision:    {metrics['precision']:.4f}")
print(f"Recall:       {metrics['recall']:.4f}")
print(f"F1-Score:     {metrics['f1_score']:.4f}")
print(f"IoU:          {metrics['iou']:.4f}")
print(f"ROC AUC:      {metrics['roc_auc']:.4f}")
print(f"Specificity:  {metrics['specificity']:.4f}")
print(f"\nConfusion Matrix:")
print(f"  TN: {tn:6d}  FP: {fp:6d}")
print(f"  FN: {fn:6d}  TP: {tp:6d}")

## Visualizations

In [ ]:
# Confusion matrix visualization
fig, ax = plt.subplots(figsize=(8, 6))
cm_display = np.array([[tn, fp], [fn, tp]])
im = ax.imshow(cm_display, cmap='Blues', aspect='auto')

# Labels
labels = ['Unburned', 'Burned']
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(labels)
ax.set_yticklabels(labels)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Ground Truth', fontsize=12)
ax.set_title('Confusion Matrix (Threshold 0.5)', fontsize=14)

# Add text annotations
for i in range(2):
    for j in range(2):
        text = ax.text(j, i, cm_display[i, j],
                      ha="center", va="center", color="w" if cm_display[i, j] > cm_display.max()/2 else "black",
                      fontsize=14, fontweight='bold')

plt.colorbar(im, ax=ax)
plt.tight_layout()
confusion_matrix_path = output_dir / 'confusion_matrix.png'
plt.savefig(confusion_matrix_path, dpi=150, bbox_inches='tight')
print(f"✓ Saved confusion matrix to {confusion_matrix_path}")
plt.show()

# ROC curve
fpr, tpr, thresholds = roc_curve(target_flat, pred_prob_flat)
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC={roc_auc:.4f})')
ax.plot([0, 1], [0, 1], 'r--', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve', fontsize=14)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
roc_path = output_dir / 'roc_curve.png'
plt.savefig(roc_path, dpi=150, bbox_inches='tight')
print(f"✓ Saved ROC curve to {roc_path}")
plt.show()

# Threshold analysis
thresholds_to_test = np.linspace(0.1, 0.9, 20)
ious_by_threshold = []
precisions_by_threshold = []
recalls_by_threshold = []

for t in thresholds_to_test:
    pred_t = (predictions > t).float().numpy().flatten()
    intersection = np.sum(pred_t * target_flat)
    union = np.sum(pred_t + target_flat) - intersection
    iou_t = intersection / (union + 1e-8)
    ious_by_threshold.append(iou_t)
    
    p, r, _, _ = precision_recall_fscore_support(target_flat, pred_t, average='binary')
    precisions_by_threshold.append(p)
    recalls_by_threshold.append(r)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(thresholds_to_test, ious_by_threshold, 'b-o', label='IoU', linewidth=2, markersize=6)
ax.plot(thresholds_to_test, precisions_by_threshold, 'g-s', label='Precision', linewidth=2, markersize=6)
ax.plot(thresholds_to_test, recalls_by_threshold, 'r-^', label='Recall', linewidth=2, markersize=6)
ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5, label='Default (0.5)')
ax.set_xlabel('Threshold', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Metrics vs Probability Threshold', fontsize=14)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.set_ylim([0, 1])
threshold_path = output_dir / 'threshold_analysis.png'
plt.savefig(threshold_path, dpi=150, bbox_inches='tight')
print(f"✓ Saved threshold analysis to {threshold_path}")
plt.show()

## Error Analysis

In [ ]:
# Identify error types per sample
pred_binary_full = (predictions > 0.5).float()
false_positives = []  # Predicted burned, actually unburned
false_negatives = []  # Predicted unburned, actually burned
true_positives = []   # Predicted burned, actually burned
true_negatives = []   # Predicted unburned, actually unburned

for i in range(len(predictions)):
    pred_sum = pred_binary_full[i].sum().item()
    target_sum = targets[i].sum().item()
    overlap = (pred_binary_full[i] * targets[i]).sum().item()
    
    if pred_sum > 0 and target_sum == 0:
        false_positives.append((i, pred_sum))
    elif pred_sum == 0 and target_sum > 0:
        false_negatives.append((i, target_sum))
    elif overlap > 0:
        true_positives.append((i, overlap))
    else:
        true_negatives.append(i)

print(f"Sample-level error distribution:")
print(f"  True Positives:  {len(true_positives)} samples")
print(f"  True Negatives:  {len(true_negatives)} samples")
print(f"  False Positives: {len(false_positives)} samples (overpredicting burn)")
print(f"  False Negatives: {len(false_negatives)} samples (underpredicting burn)")

# Visualize error samples
if len(false_positives) > 0 or len(false_negatives) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    fig.suptitle('Error Analysis: False Positives (top) & False Negatives (bottom)', fontsize=14)
    
    # False positives
    for j, (idx, _) in enumerate(false_positives[:3]):
        ax = axes[0, j]
        img = images[idx, 10].numpy()  # SWIR band
        pred = pred_binary_full[idx].numpy()
        overlay = np.stack([img, img, img])
        overlay[0][pred > 0] = 1  # Red channel for predictions
        ax.imshow(np.transpose(overlay, (1, 2, 0)))
        ax.set_title(f'FP Sample {idx}')
        ax.axis('off')
    
    # False negatives
    for j, (idx, _) in enumerate(false_negatives[:3]):
        ax = axes[1, j]
        img = images[idx, 10].numpy()  # SWIR band
        target = targets[idx].numpy()
        overlay = np.stack([img, img, img])
        overlay[0][target > 0] = 1  # Red channel for ground truth
        ax.imshow(np.transpose(overlay, (1, 2, 0)))
        ax.set_title(f'FN Sample {idx}')
        ax.axis('off')
    
    plt.tight_layout()
    error_analysis_path = output_dir / 'error_analysis.png'
    plt.savefig(error_analysis_path, dpi=150, bbox_inches='tight')
    print(f"\n✓ Saved error analysis to {error_analysis_path}")
    plt.show()

## Save Results to Google Drive

In [ ]:
# Save metrics.json
metrics_path = output_dir / 'metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"✓ Saved metrics to {metrics_path}")

# Backup all analysis outputs to Google Drive
import shutil
import glob

output_files = glob.glob(str(output_dir / '*.png')) + glob.glob(str(output_dir / '*.json'))

print(f"\nBacking up {len(output_files)} files to Google Drive...")
for file_path in output_files:
    file_name = Path(file_path).name
    drive_path = drive_results_dir / file_name
    try:
        shutil.copy(file_path, drive_path)
        print(f"  ✓ {file_name}")
    except Exception as e:
        print(f"  ✗ {file_name}: {e}")

print(f"\n✓ All results backed up to {drive_results_dir}")
print(f"\n=== Analysis Complete ===")
print(f"Local results:  {output_dir}")
print(f"Drive backup:   {drive_results_dir}")
print(f"\nKey metrics:")
print(f"  Accuracy:  {metrics['accuracy']:.4f}")
print(f"  Precision: {metrics['precision']:.4f}")
print(f"  Recall:    {metrics['recall']:.4f}")
print(f"  F1-Score:  {metrics['f1_score']:.4f}")
print(f"  IoU:       {metrics['iou']:.4f}")
print(f"  ROC AUC:   {metrics['roc_auc']:.4f}")